In [2]:
import os
import pandas as pd
import numpy as np

# 1. Automatic file search for bank-full.csv or bank.csv
target_file = None
for root, dirs, files in os.walk("."):
    if "bank-full.csv" in files:
        target_file = os.path.join(root, "bank-full.csv")
        break
    elif "bank.csv" in files and target_file is None:
        target_file = os.path.join(root, "bank.csv")

# 2. Load the detected file with semicolon separator
if target_file:
    print(f"Loading file from path: '{target_file}'")
    df_raw = pd.read_csv(target_file, sep=";")
    print("Raw Dataset Dimensions:", df_raw.shape)
    display(df_raw.head())
else:
    print("Error: Could not find 'bank-full.csv' or 'bank.csv' in this folder!")

Loading file from path: '.\da_env\bank+marketing\bank\bank-full.csv'
Raw Dataset Dimensions: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [3]:
# Create an audit table to inspect data types, nulls, and hidden 'unknown' values
data_audit = pd.DataFrame({
    'Data_Type': df_raw.dtypes,
    'Null_Count': df_raw.isna().sum(),
    'Null_Percentage': (df_raw.isna().mean() * 100).round(2),
    'Unknown_Entries': (df_raw == 'unknown').sum(),
    'Unique_Values': df_raw.nunique()
})

print("=== Dataset Health Audit ===")
data_audit

=== Dataset Health Audit ===


,Data_Type,Null_Count,Null_Percentage,Unknown_Entries,Unique_Values
age,int64,0,0.0,0,77
job,str,0,0.0,288,12
marital,str,0,0.0,0,3
education,str,0,0.0,1857,4
default,str,0,0.0,0,2
balance,int64,0,0.0,0,7168
housing,str,0,0.0,0,2
loan,str,0,0.0,0,2
contact,str,0,0.0,13020,3
day,int64,0,0.0,0,31


In [4]:
# Create a clean working copy
df_clean = df_raw.copy()

# Select object/text columns
string_cols = df_clean.select_dtypes(include=['object']).columns

# Strip whitespace and convert text to lowercase
for col in string_cols:
    df_clean[col] = df_clean[col].astype("string").str.strip().str.lower()

print("Standardized string columns:")
print(list(string_cols))
df_clean.head(3)

C:\Users\kingj\AppData\Local\Temp\ipykernel_5900\1426098305.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_cols = df_clean.select_dtypes(include=['object']).columns


Standardized string columns:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'y']


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no


In [5]:
# 1. Convert target 'y' ('yes'/'no') into binary numeric (1/0)
df_clean['subscribed'] = df_clean['y'].eq('yes').astype(int)

# 2. Flag whether client was previously contacted (1 if previous > 0, else 0)
df_clean['has_previous_contact'] = df_clean['previous'].gt(0).astype(int)

# 3. Bin account balances into 4 quartile bands
df_clean['balance_band'] = pd.qcut(
    df_clean['balance'], 
    q=4, 
    labels=['Q1_Low', 'Q2_MidLow', 'Q3_MidHigh', 'Q4_High']
)

# Inspect new engineered features
df_clean[['balance', 'balance_band', 'previous', 'has_previous_contact', 'y', 'subscribed']].head()

,balance,balance_band,previous,has_previous_contact,y,subscribed
0,2143,Q4_High,0,0,no,0
1,29,Q1_Low,0,0,no,0
2,2,Q1_Low,0,0,no,0
3,1506,Q4_High,0,0,no,0
4,1,Q1_Low,0,0,no,0


In [6]:
# Check for exact duplicate rows across the dataset
duplicate_count = df_clean.duplicated().sum()
print(f"Duplicate rows identified: {duplicate_count}")

if duplicate_count > 0:
    df_clean = df_clean.drop_duplicates().copy()
    print("Duplicates dropped successfully.")
else:
    print("No duplicate rows found. Row count remains intact.")

Duplicate rows identified: 0
No duplicate rows found. Row count remains intact.


In [7]:
# 1. Construct explicit job mapping table
job_lookup = pd.DataFrame({
    "job": ["admin.", "technician", "services", "management", "retired", 
            "blue-collar", "self-employed", "entrepreneur", "housemaid", 
            "student", "unemployed", "unknown"],
    "job_category": ["Office", "Technical", "Service", "Executive", "Retired", 
                    "Manual", "Self-Employed", "Executive", "Manual", 
                    "Student", "Unemployed", "Unknown"]
})

# 2. Pre-merge assertion: Primary key must be unique
assert job_lookup["job"].is_unique, "Lookup table key is not unique!"

# 3. Executed validated Left Join
df_enriched = df_clean.merge(
    job_lookup,
    on="job",
    how="left",
    validate="many_to_one",
    indicator=True
)

# 4. Verify merge completeness
print("=== Merge Result Verification ===")
print(df_enriched["_merge"].value_counts())
assert (df_enriched["_merge"] == "both").all(), "Error: Unmatched keys found in join!"

# Drop temporary merge indicator column post-verification
df_enriched.drop(columns=["_merge"], inplace=True)
df_enriched[['job', 'job_category', 'subscribed']].head()

=== Merge Result Verification ===
_merge
both          45211
left_only         0
right_only        0
Name: count, dtype: int64


,job,job_category,subscribed
0,management,Executive,0
1,technician,Technical,0
2,entrepreneur,Executive,0
3,blue-collar,Manual,0
4,unknown,Unknown,0


In [8]:
# 1. Aggregate conversion rates by Job Category and Contact Channel
kpi_summary = (
    df_enriched.groupby(["job_category", "contact"])
    .agg(
        total_contacts=("subscribed", "size"),
        subscriptions=("subscribed", "sum"),
        conversion_rate=("subscribed", "mean"),
        avg_balance=("balance", "mean")
    )
    .reset_index()
)

# Convert conversion rate to percentage
kpi_summary["conversion_pct"] = (kpi_summary["conversion_rate"] * 100).round(2)

# Filter out low-sample buckets (< 50 contacts) and sort descending
top_segments = kpi_summary.query("total_contacts >= 50").sort_values("conversion_pct", ascending=False)
print("=== Top Converting Segments ===")
display(top_segments.head(10))

# 2. Pre-export data integrity assertions
assert len(df_enriched) == len(df_clean), "Row count discrepancy detected post-merge!"
assert df_enriched['subscribed'].isin([0, 1]).all(), "Target variable contains invalid values!"

# 3. Export clean dataset to CSV
output_filename = "cleaned_bank_marketing_final.csv"
df_enriched.to_csv(output_filename, index=False)
print(f"\nSuccess! Dataset cleaned, validated, and saved to '{output_filename}'.")

=== Top Converting Segments ===


,job_category,contact,total_contacts,subscriptions,conversion_rate,avg_balance,conversion_pct
18,Student,cellular,697,246,0.352941,1497.230990,35.29
10,Retired,telephone,405,129,0.318519,2687.274074,31.85
9,Retired,cellular,1370,366,0.267153,1957.398540,26.72
27,Unknown,cellular,131,27,0.206107,1954.083969,20.61
24,Unemployed,cellular,891,173,0.194164,1633.041526,19.42
19,Student,telephone,99,19,0.191919,1110.434343,19.19
6,Office,cellular,3354,529,0.157722,1113.124031,15.77
0,Executive,cellular,7975,1243,0.155862,1748.999373,15.59
7,Office,telephone,289,41,0.141869,1689.965398,14.19
12,Self-Employed,cellular,1084,153,0.141144,1684.715867,14.11



Success! Dataset cleaned, validated, and saved to 'cleaned_bank_marketing_final.csv'.


In [9]:
import os

full_path = os.path.abspath("cleaned_bank_marketing_final.csv")
print("File location on your computer:")
print(full_path)

File location on your computer:
C:\Users\kingj\Documents\Data-Science\data-acqusition\cleaned_bank_marketing_final.csv
